# Part 1: Data Exploration and Preprocessing

In this notebook, you will implement functions to load, preprocess, and visualize physiological data from the Wearable Exam Stress Dataset.

In [9]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import os
from scipy.stats import zscore

# Set style for plots
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Data Loading

Implement the `load_data` function to read and organize the physiological data from the dataset.

In [10]:
def load_data(data_dir='data/raw'):
    """Load and organize the physiological data from the dataset.
    
    Parameters
    ----------
    data_dir : str
        Path to the directory containing the dataset files
        
    Returns
    -------
    pd.DataFrame
        DataFrame containing the organized physiological data with columns:
        ['timestamp', 'heart_rate', 'eda', 'temperature', 'subject_id', 'session']
    """
    # Your code here 
    hr_path = os.path.join(data_dir, 'HR.csv')
    with open(hr_path, 'r') as f:
        lines = f.readlines()
        start_time_hr = float(lines[0].strip())
        sample_rate_hr = float(lines[1].strip())
        values_hr = [float(line.strip()) for line in lines[2:]]
        timestamps_hr = [start_time_hr + i / sample_rate_hr for i in range(len(values_hr))]
        df_hr = pd.DataFrame({'timestamp': timestamps_hr, 'heart_rate': values_hr})
    eda_path = os.path.join(data_dir, 'EDA.csv')
    with open(eda_path, 'r') as f:
        lines = f.readlines()
        start_time_eda = float(lines[0].strip())
        sample_rate_eda = float(lines[1].strip())
        values_eda = [float(line.strip()) for line in lines[2:]]
        timestamps_eda = [start_time_eda + i / sample_rate_eda for i in range(len(values_eda))]
        df_eda = pd.DataFrame({'timestamp': timestamps_eda, 'eda': values_eda})
    temp_path = os.path.join(data_dir, 'TEMP.csv')
    with open(temp_path, 'r') as f:
        lines = f.readlines()
        start_time_temp = float(lines[0].strip())
        sample_rate_temp = float(lines[1].strip())
        values_temp = [float(line.strip()) for line in lines[2:]]
        timestamps_temp = [start_time_temp + i / sample_rate_temp for i in range(len(values_temp))]
        df_temp = pd.DataFrame({'timestamp': timestamps_temp, 'temperature': values_temp})
    df = pd.merge(df_hr, df_eda, on='timestamp', how='outer')
    df = pd.merge(df, df_temp, on='timestamp', how='outer')
    df['subject_id'] = os.path.basename(data_dir)
    df['session'] = 'session1'  
    df = df.sort_values(by='timestamp').reset_index(drop=True)
    return df[['timestamp', 'heart_rate', 'eda', 'temperature', 'subject_id', 'session']]

## 2. Data Preprocessing

Implement the `preprocess_data` function to clean and prepare the data for analysis.

In [11]:
def preprocess_data(data, output_dir='data/processed'):
    """Clean and prepare the physiological data for analysis.
    
    Parameters
    ----------
    data : pd.DataFrame
        Raw physiological data
    output_dir : str
        Directory to save processed data files
        
    Returns
    -------
    pd.DataFrame
        Cleaned and preprocessed data
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Your code here
    # 1. Handle missing values
    data = data.sort_values('timestamp')
    data = data.fillna(method='ffill').fillna(method='bfill')
    # 2. Resample to regular intervals
    data['datetime'] = pd.to_datetime(data['timestamp'], unit='s')
    data = data.set_index('datetime')
    resampled = data.resample('1S').mean().interpolate()
    # 3. Remove outliers (z-score > 3)
    for col in ['heart_rate', 'eda', 'temperature']:
        if col in resampled.columns:
            z = np.abs(zscore(resampled[col], nan_policy='omit'))
            resampled[col] = resampled[col].where(z <= 3)
    resampled = resampled.interpolate()
    resampled['timestamp'] = resampled.index.astype(np.int64) // 10**9
    resampled['subject_id'] = data['subject_id'].iloc[0]
    resampled['session'] = data['session'].iloc[0]
    # 4. Save processed data to CSV files
    output_path = os.path.join(output_dir, f"{resampled['subject_id'].iloc[0]}_{resampled['session'].iloc[0]}.csv")
    resampled.to_csv(output_path, index=False)

    return resampled.reset_index(drop=True)

## 3. Visualization

Implement the `plot_physiological_signals` function to create visualizations of the physiological data.

In [13]:
def plot_physiological_signals(data, subject_id, session, output_dir='plots'):
    """Create plots of physiological signals for a given subject and session.
    
    Parameters
    ----------
    data : pd.DataFrame
        Preprocessed physiological data
    subject_id : str
        Subject identifier (e.g., 'S1')
    session : str
        Session identifier (e.g., 'Midterm 1')
    output_dir : str
        Directory to save plot files
        
    Returns
    -------
    matplotlib.figure.Figure
        Figure object containing the plots
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    df = data[(data['subject_id'] == subject_id) & (data['session'] == session)]
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(f'Physiological Signals for {subject_id} - {session}', fontsize=16)
    axs[0].plot(df['datetime'], df['heart_rate'], color='red')
    axs[0].set_ylabel('Heart Rate (bpm)')
    axs[0].set_title('Heart Rate')
    axs[1].plot(df['datetime'], df['eda'], color='blue')
    axs[1].set_ylabel('EDA (μS)')
    axs[1].set_title('Electrodermal Activity')
    axs[2].plot(df['datetime'], df['temperature'], color='green')
    axs[2].set_ylabel('Temp (°C)')
    axs[2].set_title('Temperature')
    axs[2].set_xlabel('Time')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    filename = f"{subject_id}_{session.replace(' ', '_')}.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    return fig